# ACE-RAG Kaggle Runbook

Upload the latest `ace_rag_research` folder or zip once. Then use this notebook to run all staged experiments without reuploading for each run.

Recommended Kaggle settings: GPU `T4 x2`, Internet `On`.

## 1. Locate Project

This cell handles either an uploaded folder or any uploaded zip file containing `ace_rag_research`.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
work_root = Path('/kaggle/working')
dst = work_root / 'ace_rag_research'

if dst.exists():
    shutil.rmtree(dst)

folder_candidates = [
    p for p in input_root.glob('**/ace_rag_research')
    if (p / 'ace_rag').exists() and (p / 'experiments').exists()
]

if folder_candidates:
    src = folder_candidates[0]
    print('Found project folder:', src)
    shutil.copytree(src, dst)
else:
    zip_candidates = list(input_root.glob('**/*.zip'))
    print('Zip candidates:', zip_candidates)
    if not zip_candidates:
        raise FileNotFoundError('No ace_rag_research folder or zip found under /kaggle/input')
    zip_path = zip_candidates[0]
    temp = work_root / '_ace_extract'
    if temp.exists():
        shutil.rmtree(temp)
    temp.mkdir(parents=True, exist_ok=True)
    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(temp)
    extracted_candidates = [
        p for p in temp.glob('**/ace_rag_research')
        if (p / 'ace_rag').exists() and (p / 'experiments').exists()
    ]
    if extracted_candidates:
        shutil.copytree(extracted_candidates[0], dst)
    elif (temp / 'ace_rag').exists() and (temp / 'experiments').exists():
        shutil.copytree(temp, dst)
    else:
        raise FileNotFoundError('Zip extracted, but project folder was not found')

os.chdir(dst)
print('Now in:', Path.cwd())
!ls

## 2. Install And Check GPU

In [ ]:
!python -m pip install -q -r requirements-cloud.txt
!python -m scripts.cloud_check
!nvidia-smi

## 3. Direct Sentence-Transformer GPU Test

If this does not show `device=cuda`, do not start the big experiments.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('cuda_device_count:', torch.cuda.device_count())

model = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda')
print('sentence-transformer device:', model.device)
emb = model.encode(['gpu test sentence'] * 512, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print('embedding shape:', emb.shape)
!nvidia-smi

## 4. Toy Smoke Test

This should finish quickly. It checks imports, configs, graph build, retrieval, and metric writing.

In [ ]:
!python -m experiments.run_mvp \
  --config configs/toy.yaml \
  --methods ace_graph \
  --no-save-runs \
  --out-dir cloud_results

## 5. HotpotQA Baseline, Limit 200

Run this once per fresh session. It gives chunk baseline + ACE comparison.

In [ ]:
!python -m experiments.run_mvp \
  --config configs/hotpotqa.yaml \
  --limit 200 \
  --device cuda \
  --no-save-runs \
  --out-dir cloud_results

## 6. Best Current ACE Run, Limit 1000

This skips chunk and no-conflict variants. Use this for tuning.

In [ ]:
!python -m experiments.run_mvp \
  --config configs/hotpotqa.yaml \
  --limit 1000 \
  --compressor identity \
  --top-k-nodes 48 \
  --max-expanded-docs 6 \
  --methods ace_graph \
  --device cuda \
  --no-save-runs \
  --out-dir cloud_results

## 7. Optional ACE Budget Sweep

Run these only after the previous cell finishes. They test whether a slightly larger document budget recovers recall.

In [ ]:
for docs in [5, 6, 7]:
    print('\n=== identity docs', docs, '===')
    !python -m experiments.run_mvp \
      --config configs/hotpotqa.yaml \
      --limit 1000 \
      --compressor identity \
      --top-k-nodes 48 \
      --max-expanded-docs {docs} \
      --methods ace_graph \
      --device cuda \
      --no-save-runs \
      --out-dir cloud_results

## 8. Optional Compression Sweep

These are not expected to beat identity yet. They test whether compression can preserve enough retrieval quality.

In [ ]:
compression_runs = [
    ('pca', 256, 48, 8),
    ('truncate', 256, 48, 6),
    ('truncate', 384, 48, 6),
]

for compressor, dims, nodes, docs in compression_runs:
    print('\n===', compressor, dims, 'nodes', nodes, 'docs', docs, '===')
    !python -m experiments.run_mvp \
      --config configs/hotpotqa.yaml \
      --limit 1000 \
      --compressor {compressor} \
      --compress-dims {dims} \
      --top-k-nodes {nodes} \
      --max-expanded-docs {docs} \
      --methods ace_graph \
      --device cuda \
      --no-save-runs \
      --out-dir cloud_results

## 9. Print Metrics

In [ ]:
from pathlib import Path
import pandas as pd

paths = sorted(Path('cloud_results').glob('*hotpotqa*metrics.csv'))
print('metric files:', len(paths))
for path in paths:
    print('\n---', path.name, '---')
    df = pd.read_csv(path)
    display(df)

rows = []
for path in paths:
    df = pd.read_csv(path)
    df.insert(0, 'file', path.name)
    rows.append(df)
if rows:
    all_df = pd.concat(rows, ignore_index=True)
    cols = ['file', 'method', 'compressor', 'recall@5', 'all_gold@5', 'avg_evidence_tokens', 'avg_expanded_docs', 'avg_hits', 'extractive_f1']
    display(all_df[[c for c in cols if c in all_df.columns]].sort_values(['recall@5', 'avg_evidence_tokens'], ascending=[False, True]))

## 10. Zip Results For Download

In [ ]:
!zip -r ace_rag_cloud_results.zip cloud_results
!ls -lh ace_rag_cloud_results.zip

## 11. Stage 2: Free Qwen Reader Evaluation

Run this after retrieval tuning. Start with a small limit because local generation is slower than retrieval. This compares Chunk RAG and ACE-RAG using the same free Qwen reader.

In [ ]:
!python -m experiments.run_qwen_eval \
  --dataset hotpotqa \
  --limit 50 \
  --methods chunk,ace_graph \
  --compressor truncate \
  --compress-dims 320 \
  --reader-model Qwen/Qwen2.5-1.5B-Instruct \
  --reader-device cuda \
  --reader-batch-size 4 \
  --reader-context sources \
  --out-dir cloud_results

## 12. Print Qwen Metrics

In [ ]:
!ls -lh cloud_results/*qwen*
!cat cloud_results/*qwen*metrics.csv

## 13. Stage 2B: Qwen With Snippet Context

This is the intended ACE reader format: retrieve graph nodes, expand to short source snippets, and avoid feeding full passages.

In [ ]:
!python -m experiments.run_qwen_eval \
  --dataset hotpotqa \
  --limit 200 \
  --methods chunk,ace_graph \
  --compressor truncate \
  --compress-dims 320 \
  --reader-model Qwen/Qwen2.5-1.5B-Instruct \
  --reader-device cuda \
  --reader-batch-size 4 \
  --reader-context snippets \
  --snippet-window 1 \
  --max-snippets 8 \
  --max-snippet-tokens 80 \
  --out-dir cloud_results